# Prep balanced dataset after CTGAN/tabDDPM augmentation

This notebook:
- reads `<model>/Balanced/train_input_with_label.csv`
- reads `<model>/Balanced/synthetic_*_combined.csv`
- verifies the columns match
- merges, shuffles, and writes the combined dataset


In [ ]:
from pathlib import Path

import pandas as pd

## Config

In [ ]:
MODEL_CONFIGS = {
    "ctgan": {
        "root": Path("5G-NIDD_Aug_CTGAN/Balanced"),
        "synthetic_filename": "synthetic_ctgan_combined.csv",
    },
    "tabddpm": {
        "root": Path("5G-NIDD_Aug_tabDDPM/Balanced"),
        "synthetic_filename": "synthetic_tabddpm_combined.csv",
    },
}
MODELS_TO_RUN = ["ctgan", "tabddpm"]  # edit this list to run one model

TRAIN_FILENAME = "train_input_with_label.csv"

OUT_ROOT = Path("Datasets/augmented_balanced_datasets")

RANDOM_SEED = 42


## Helpers

In [ ]:
def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)

def validate_and_align_columns(train_df: pd.DataFrame, synth_df: pd.DataFrame) -> pd.DataFrame:
    if list(train_df.columns) == list(synth_df.columns):
        return synth_df

    train_cols = list(train_df.columns)
    synth_cols = list(synth_df.columns)

    missing_in_synth = [c for c in train_cols if c not in synth_cols]
    extra_in_synth = [c for c in synth_cols if c not in train_cols]

    if missing_in_synth or extra_in_synth:
        msg = (
            "Column mismatch between train and synthetic. "
            f"Missing in synthetic: {missing_in_synth}; Extra in synthetic: {extra_in_synth}"
        )
        raise ValueError(msg)

    print("Columns match but order differs; aligning synthetic to train order.")
    return synth_df.reindex(columns=train_cols)


## Merge

In [ ]:
for model_name in MODELS_TO_RUN:
    cfg = MODEL_CONFIGS.get(model_name)
    if not cfg:
        print(f"Skip model '{model_name}': not in MODEL_CONFIGS")
        continue

    balanced_dir = cfg["root"]
    synth_filename = cfg["synthetic_filename"]

    train_path = balanced_dir / TRAIN_FILENAME
    synth_path = balanced_dir / synth_filename

    if not train_path.exists():
        print(f"[{model_name}] Missing train file: {train_path}")
        continue
    if not synth_path.exists():
        print(f"[{model_name}] Missing synthetic file: {synth_path}")
        continue

    print(f"[{model_name}] Train path: {train_path}")
    print(f"[{model_name}] Synthetic path: {synth_path}")

    train_df = pd.read_csv(train_path)
    synth_df = pd.read_csv(synth_path)

    synth_df = validate_and_align_columns(train_df, synth_df)

    combined_df = pd.concat([train_df, synth_df], ignore_index=True)
    combined_df = combined_df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

    out_dir = OUT_ROOT / model_name
    ensure_dir(out_dir)
    out_path = out_dir / "mixed_augmented_balanced.csv"
    combined_df.to_csv(out_path, index=False)

    print(f"[{model_name}] Saved -> {out_path} | train {train_df.shape} + synth {synth_df.shape} = combined {combined_df.shape}")
